# Risk scoring + Explainability + Random Forest

Owner: Thanthrige D.P

Put `PhiUSIIL_Phishing_URL_Dataset.csv` in `data/raw/` before running.

## To do

- [ ] Train, tune and evaluate the Random Forest in `src/models/forest.py`
- [ ] Risk score from the phishing probability, risk bands (4.11)
- [ ] Calibration check: calibration curve / Brier score before and after
- [ ] SHAP global importance and summary plot (4.13)
- [ ] Explain single predictions: score + class + top features
- [ ] Final analysis across all parts (4.14)

## Setup

In [ ]:
import sys
sys.path.append("..")

import pandas as pd
import matplotlib.pyplot as plt

from src import config, data, features, evaluate, risk, explain
from src.models import logreg, tree, svm, forest

# Work on a sample first; set SAMPLE = None for the full dataset.
SAMPLE = 20000

df = data.load_clean_data()
if SAMPLE and SAMPLE < len(df):
    df = df.sample(n=SAMPLE, random_state=config.RANDOM_STATE).reset_index(drop=True)
df.shape

## Phase 1: Random Forest (proposal 4.9)

Cross-validation on the training set only (`evaluate.cv_splitter()`, 5 stratified folds).
The test set is not touched here.

In [ ]:
import warnings
warnings.filterwarnings("ignore", message=".*sklearn.utils.parallel.delayed.*")  # harmless joblib notice
from sklearn.ensemble import RandomForestClassifier

original = data.get_original_features(df)
df = features.add_features(df)
candidates = original + features.ENGINEERED_FEATURES
X_train_all, _, _, _ = data.split(df, candidates)
# Feature selection sees training rows only.
engineered = features.select_features(df.loc[X_train_all.index], candidates)
feature_sets = {"original": original, "engineered": engineered}
{name: len(cols) for name, cols in feature_sets.items()}

In [ ]:
SHOW = ["f1_mean", "f1_std", "roc_auc_mean", "recall_mean", "precision_mean", "accuracy_mean"]

def cv_table(make_model):
    """CV metrics of make_model() on each feature set (training rows only)."""
    rows = {}
    for name, cols in feature_sets.items():
        X_tr, _, y_tr, _ = data.split(df, cols)
        rows[name] = evaluate.cv_evaluate(make_model(), X_tr, y_tr)
    return pd.DataFrame(rows).T[SHOW]

def default_forest():
    """Baseline: sklearn defaults with 200 trees (the starting build())."""
    return RandomForestClassifier(n_estimators=200, n_jobs=-1, random_state=config.RANDOM_STATE)

baseline_cv = cv_table(default_forest)
baseline_cv.round(4)

### Hyperparameter search

`forest.tune()`: RandomizedSearchCV, 20 candidates, shared CV folds, chosen by F1,
on the original-feature training set. One set of values goes into `build()`,
because `main.py` uses the same `build()` for both feature sets.

In [ ]:
X_train, X_test, y_train, y_test = data.split(df, original)
search, seconds = forest.tune(X_train, y_train, n_iter=20, scoring="f1")
print(f"Search took {seconds/60:.1f} min for {len(search.cv_results_['params'])} candidates x {config.CV_FOLDS} folds")

# Full search results to a file; only the top rows are shown here.
out = config.RESULTS_DIR / (f"results_sample_{SAMPLE}_forest_search.csv" if SAMPLE else "forest_search.csv")
pd.DataFrame(search.cv_results_).to_csv(out, index=False)
print("best:", search.best_params_)
forest.search_table(search, top=8)

### Tuned model vs baseline

The search is flat: six candidates tie for best CV F1 (0.99993, std 0.00015), and every
other candidate is one URL worse over all folds, which is within noise. `best_params_`
just reports the first of the tied six. Tie-break rule: take the cheapest and most
regularised tied candidate (100 trees, max_depth 30, min_samples_leaf 8, sqrt features),
since extra trees buy nothing here and larger leaves give smoother probabilities for the
risk score. These values are written into `forest.build()`. Provisional until the
leakage check (phase 2): scores this close to 1.0 are a warning, not a result.


In [ ]:
tuned_cv = cv_table(forest.build)
comparison = pd.concat({"baseline": baseline_cv, "tuned": tuned_cv}).round(4)
comparison

## Phase 2: Leakage check

CV F1 of 0.9999 (1 error in 16,000 URLs) is a warning, not a result. This section looks for
features that give the label away, using **training rows only** (`X_train`, `y_train` from the
original-feature split above). The test set is still untouched.

Three checks:
1. **Single-feature ROC-AUC.** Each feature is used directly as a score, with no model fitted.
   `sep_auc = max(AUC, 1 - AUC)`, so 1.0 means that feature alone separates the classes perfectly.
2. **Class-constant features.** These take one value for every URL in one class, which points to
   how the dataset was collected rather than to anything about phishing.
3. **Ablation.** CV of the tuned forest after removing suspect features or feature groups.

This section only measures. The group decides what to drop (`features.select_features()`, owner Madushan S.C).

In [ ]:
from sklearn.metrics import roc_auc_score

def single_feature_auc(X, y):
    """ROC-AUC of each column used directly as a score (no fitting), sorted by separability."""
    rows = []
    for c in X.columns:
        auc = roc_auc_score(y, X[c])
        rows.append({"feature": c, "auc": auc, "sep_auc": max(auc, 1 - auc),
                     "n_unique": X[c].nunique(),
                     "mean_phishing": X.loc[y == 1, c].mean(),
                     "mean_legit": X.loc[y == 0, c].mean()})
    return pd.DataFrame(rows).sort_values("sep_auc", ascending=False).reset_index(drop=True)

auc_table = single_feature_auc(X_train, y_train)
auc_out = config.RESULTS_DIR / (f"results_sample_{SAMPLE}_single_feature_auc.csv" if SAMPLE else "single_feature_auc.csv")
auc_table.to_csv(auc_out, index=False)
auc_table.head(12).round(4)

In [ ]:
top = auc_table.head(20).iloc[::-1]
fig, ax = plt.subplots(figsize=(7, 6))
ax.barh(top["feature"], top["sep_auc"])
ax.axvline(0.95, color="red", ls="--", lw=1, label="0.95")
ax.set_xlim(0.5, 1.0)
ax.set_xlabel("single-feature ROC-AUC, max(AUC, 1 - AUC), training rows")
ax.set_title("How well each feature separates the classes on its own")
ax.legend(loc="lower right")
config.FIGURES_DIR.mkdir(parents=True, exist_ok=True)
fig.savefig(config.FIGURES_DIR / "rf_single_feature_auc.png", dpi=150, bbox_inches="tight")
plt.show()

**Reading the table:** `URLSimilarityIndex` alone reaches about 0.996. The next six (`LineOfCode`,
`NoOfExternalRef`, `NoOfImage`, `NoOfSelfRef`, `NoOfJS`, `NoOfCSS`) are page-content counts, each
around 0.96 to 0.99: in this dataset, phishing pages are almost empty (mean `LineOfCode` about 70
compared with about 1,900 for legitimate pages). That is probably because many phishing pages were
already taken down or were bare when the dataset was crawled. They are real signals, but they come
partly from how the dataset was collected.

In [ ]:
def class_constant_features(X, y):
    """Features that take one single value for every row of one class."""
    rows = []
    for c in X.columns:
        for cls, name in ((0, "legit"), (1, "phishing")):
            v = X.loc[y == cls, c]
            if v.nunique() == 1:
                val = v.iloc[0]
                rows.append({"feature": c, "constant_in": name, "value": val,
                             "other_class_share_same_value": (X.loc[y != cls, c] == val).mean()})
    return pd.DataFrame(rows)

display(class_constant_features(X_train, y_train).round(4))

# A one-line rule with no model: "URLSimilarityIndex below 100 means phishing".
from sklearn.metrics import confusion_matrix, f1_score
rule = (X_train["URLSimilarityIndex"] < 100).astype(int)
tn, fp, fn, tp = confusion_matrix(y_train, rule).ravel()
print(f"Rule USI < 100 on training rows: F1 {f1_score(y_train, rule):.4f}  (tn {tn}, fp {fp}, fn {fn}, tp {tp})")

**`URLSimilarityIndex` is 100 for every legitimate URL.** Under PhiUSIIL's definition, this
feature measures similarity to a list of known legitimate URLs, and the legitimate URLs come from
that same list. A threshold rule with no model reaches F1 of about 0.996 with zero false positives.
This is label leakage by construction: a new legitimate URL that is not on the list would not get
100, so the feature would not behave the same way in deployment.

`IsHTTPS` is also 1 for every legitimate URL (but only for half of the phishing ones), and every
legitimate URL has no `?`, `=`, `&`, IP address or obfuscation. So the legitimate class looks like
a set of clean HTTPS homepages. This is collection bias, not leakage, but it does make the task
easier than the real one.

In [ ]:
# Feature groups from the PhiUSIIL description: computed from the URL string vs from the fetched page.
URL_FEATURES = ["URLLength", "DomainLength", "IsDomainIP", "URLSimilarityIndex", "CharContinuationRate",
                "TLDLegitimateProb", "URLCharProb", "TLDLength", "NoOfSubDomain", "HasObfuscation",
                "NoOfObfuscatedChar", "ObfuscationRatio", "NoOfLettersInURL", "LetterRatioInURL",
                "NoOfDegitsInURL", "DegitRatioInURL", "NoOfEqualsInURL", "NoOfQMarkInURL",
                "NoOfAmpersandInURL", "NoOfOtherSpecialCharsInURL", "SpacialCharRatioInURL", "IsHTTPS"]
PAGE_FEATURES = [c for c in original if c not in URL_FEATURES]
assert set(URL_FEATURES) <= set(original)

strong = auc_table.loc[auc_table["sep_auc"] >= 0.95, "feature"].tolist()
without = lambda cols, drop: [c for c in cols if c not in drop]
variants = {
    "all original": original,
    "- URLSimilarityIndex": without(original, ["URLSimilarityIndex"]),
    f"- sep_auc >= 0.95 ({len(strong)})": without(original, strong),
    "URL only": URL_FEATURES,
    "URL only - USI": without(URL_FEATURES, ["URLSimilarityIndex"]),
    "URL only - USI - IsHTTPS": without(URL_FEATURES, ["URLSimilarityIndex", "IsHTTPS"]),
    "page only": PAGE_FEATURES,
}
ablation = {}
for name, cols in variants.items():
    r = evaluate.cv_evaluate(forest.build(), X_train[cols], y_train)
    ablation[name] = {"n_features": len(cols)} | {k: r[k] for k in SHOW}
ablation = pd.DataFrame(ablation).T
ablation.round(4)

**What the ablation shows:**
- Dropping `URLSimilarityIndex` costs very little, because the page-content counts still separate the
  classes. So the near-perfect score does not come from one leaking column alone.
- Even without all seven strongest features, CV F1 stays around 0.997. The dataset is genuinely easy
  for a forest once many weak signals are combined.
- With **URL features only** (what a URL-only risk scorer would see before fetching the page), removing
  `URLSimilarityIndex` and `IsHTTPS` brings F1 down to about 0.96. That is the more honest view of
  how hard the URL part of the task is.

In [ ]:
# Near-duplicate check: clean() removes repeated URLs, but the same domain can still be on both sides.
dom = df["Domain"]
in_train = dom.loc[X_test.index].isin(set(dom.loc[X_train.index]))
print(f"Test rows whose domain also appears in training: {in_train.mean():.2%} of {len(in_train)}")

About 4% of test URLs share a domain with a training URL. That is small, but a grouped split by
domain would remove it completely. That would be a change to `data.split()` (owner Madushanka D.M.C)
and needs group agreement.

### Phase 2 outcome (for the group to decide)

| Finding | Kind | Proposal for the group |
|---|---|---|
| `URLSimilarityIndex` = 100 for every legitimate URL; one-line rule F1 about 0.996 | Label leakage by construction | Drop it in `select_features()` |
| `IsHTTPS` = 1 for every legitimate URL | Collection bias | Keep, but report it; show results with and without it |
| Page-content counts (`LineOfCode`, `NoOf*Ref`, `NoOfImage`, `NoOfJS`, `NoOfCSS`) each have AUC of 0.96 or higher | Collection bias (bare or taken-down phishing pages) | Keep, but report a URL-only result next to the full one |
| About 4% of test domains also appear in training | Mild near-duplicate overlap | Optional: grouped split by domain |

The forest hyperparameters in `build()` stay provisional until the group decides on the feature list.
After that, rerun `forest.tune()` on the agreed features.

## Final test-set report

Run once, after tuning is finished. Do not choose models or hyperparameters from this result.

In [ ]:
# Phase 5 only: run once on the full data after all choices are made.
RUN_FINAL_TEST = False
if RUN_FINAL_TEST:
    model = forest.train(X_train, y_train)
    display(evaluate.evaluate(model, X_test, y_test))

## Experiment log

| Date | What changed | Why | CV or test | Key metrics (F1 / ROC-AUC / recall) | Commit |
|---|---|---|---|---|---|
| 2026-11-17 | Baseline RF (200 trees, sklearn defaults), original (50) and engineered (52) features | Starting point (4.9) | CV, 5 folds, 20k sample | orig 0.9999 / 1.0000 / 0.9997; eng 0.9999 / 1.0000 / 0.9999 | phase 1 |
| 2026-11-17 | RandomizedSearchCV, 20 candidates, scored on F1, original features | Tune RF hyperparameters on train only | CV, 20k sample | 6-way tie at F1 0.99993; all 20 within 0.0001; 1.5-3.1 min | phase 1 |
| 2026-11-17 | build(): 100 trees, max_depth 30, min_samples_leaf 8, sqrt | Cheapest, most regularised tied candidate | CV, 20k sample | orig 0.9999 / 1.0000 / 0.9999; eng 0.9999 / 1.0000 / 0.9997; 1 CV error in 16,000 | phase 1 |
| 2026-11-17 | Single-feature ROC-AUC + class-constant check, original features | Leakage check (phase 2) | Train rows, 20k sample, no model | USI sep_auc 0.996, =100 for all legit; rule USI<100 F1 0.9958; 6 page counts sep_auc 0.96-0.99 | phase 2 |
| 2026-11-17 | Ablation: drop USI / drop 7 strongest / URL-only / page-only | How much the score depends on suspect features | CV, 20k sample | -USI 0.9979 / 1.0000 / 0.9966; -7 strongest 0.9974; URL-only -USI -IsHTTPS 0.9639 / 0.9961 / 0.9523; page-only 0.9932 | phase 2 |